# 전투 적합 여부·회복기간 예측
시작: 2026-10-09 14:45 KST · 종료: 16:35 KST
문제·채점 기준은 README.md를 확인하세요. 아래 기본 예제를 개선하고 submission.csv를 저장하세요.
서버 전체에서 커널 하나만 사용합니다. 완료 후 Kernel → Shutdown으로 실행 자리를 반납하세요.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score
DATA=Path('data')
train=pd.read_csv(DATA/'train.csv'); test=pd.read_csv(DATA/'test.csv'); sample=pd.read_csv(DATA/'sample_submission.csv')
print(train.shape, test.shape)


(80, 7) (21, 5)


## 풀이: 네 기준을 모두 통과하면 fit (상자 규칙) + status 별 회복일 평균
- **데이터 구조**: fit 은 10/80(12.5%)뿐이고, fit 10명은 모두 `pushup >= 38`, `run_time <= 9.6`, `sleep_quality >= 5.2`, `stress_level <= 5.8` 안에 있으며 이 상자 안에 unfit 은 0명입니다. 선형 경계(로지스틱)보다 "네 기준 모두 통과" 규칙이 맞는 형태입니다.
- **회복일**: status 가 거의 결정합니다(fit 평균 5.5일, unfit 평균 10.1일). unfit 안에서는 네 특징과의 상관이 -0.20~0.09 로 약하고, 기준 미달 개수별 평균도 9.5~10.6 으로 차이가 작아 **status 별 평균**만 씁니다.
- **검증**(Stratified 5-Fold x 20회 반복): 제공 베이스라인 RF 정확도 0.937, 로지스틱 0.9475 / 회복 RMSE 2.24 / 점수 73.6, **상자 규칙 0.967 / 2.17 / 74.9**(20회 중 18회에서 로지스틱보다 높음). 정답 status 를 알 때의 회복 RMSE 는 1.97 이 하한입니다.
- **주의**: 임계값은 학습 fit 10명의 극값이라 표본이 적습니다. 테스트에 그 극값보다 조금 바깥의 fit 이 있으면 unfit 으로 잘못 분류합니다. 극값과 가장 가까운 unfit 사이의 중간으로 여유를 주는 변형은 점수가 74.5 로 차이가 노이즈 수준이라 쓰지 않았습니다.

In [2]:
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

features = [c for c in test.columns if c != "soldier_id"]          # pushup, run_time, sleep_quality, stress_level
SIGN = np.array([1, -1, 1, -1])                                    # fit 방향: pushup↑, run_time↓, sleep_quality↑, stress_level↓
assert features == ["pushup", "run_time", "sleep_quality", "stress_level"]

class Box:
    """네 기준을 모두 통과하면 fit. 임계값 = 학습 데이터 fit 의 극값"""
    def fit(self, X, y): self.th = (np.asarray(X, float) * SIGN)[np.asarray(y, bool)].min(0); return self
    def predict(self, X): return ((np.asarray(X, float) * SIGN) >= self.th).all(1)

X = train[features]; is_fit = (train.status == "fit").values; rec = train.recovery_days.values.astype(float)
R = 20; cv = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=R, random_state=0).split(X, is_fit))
def evaluate(make):
    acc, rm = [], []
    for r in range(R):
        ps = np.zeros(len(train), bool); oof = np.zeros(len(train))
        for a, b in cv[r * 5:(r + 1) * 5]:
            pb = np.asarray(make().fit(X.iloc[a], is_fit[a]).predict(X.iloc[b])).astype(bool); ps[b] = pb
            oof[b] = np.where(pb, rec[a][is_fit[a]].mean(), rec[a][~is_fit[a]].mean())
        acc.append(np.mean(ps == is_fit)); rm.append(np.sqrt(np.mean((rec - oof) ** 2)))
    acc, rm = np.array(acc), np.array(rm); return acc, rm, 50 * acc + 50 / (1 + rm / 2.46475151)
print("fit 비율: %.3f (다수 클래스만 찍으면 정확도 %.3f)" % (is_fit.mean(), 1 - is_fit.mean()))
res = {"로지스틱 C=1": evaluate(lambda: make_pipeline(StandardScaler(), LogisticRegression(C=1, max_iter=2000))), "상자 규칙 (메인)": evaluate(Box)}
for n, (acc, rm, sc) in res.items(): print(f"{n:16s} 정확도 {acc.mean():.4f} | 회복 RMSE {rm.mean():.4f} | 점수 {sc.mean():.2f} (반복 sd {sc.std():.2f})")
print("상자 규칙이 로지스틱보다 높은 반복:", int((res["상자 규칙 (메인)"][2] > res["로지스틱 C=1"][2]).sum()), "/", R)

box = Box().fit(X, is_fit)
print("임계값: pushup >= %.0f, run_time <= %.1f, sleep_quality >= %.1f, stress_level <= %.1f" % tuple(box.th * SIGN))
pred_fit = box.predict(test[features]); mean_fit, mean_unfit = rec[is_fit].mean(), rec[~is_fit].mean()
submission = pd.DataFrame({"soldier_id": test.soldier_id, "status": np.where(pred_fit, "fit", "unfit"), "recovery_days": np.where(pred_fit, mean_fit, mean_unfit)})[sample.columns]
assert set(submission.status) <= {"fit", "unfit"} and (submission.recovery_days >= 0).all() and np.isfinite(submission.recovery_days).all()
submission.to_csv("submission.csv", index=False)
print("회복일 평균: fit %.2f, unfit %.2f | 테스트 fit 예측 %d/%d (%.3f, train %.3f)" % (mean_fit, mean_unfit, pred_fit.sum(), len(test), pred_fit.mean(), is_fit.mean())); submission.head()

fit 비율: 0.125 (다수 클래스만 찍으면 정확도 0.875)


로지스틱 C=1         정확도 0.9475 | 회복 RMSE 2.2410 | 점수 73.57 (반복 sd 0.85)
상자 규칙 (메인)       정확도 0.9669 | 회복 RMSE 2.1718 | 점수 74.93 (반복 sd 0.63)
상자 규칙이 로지스틱보다 높은 반복: 18 / 20
임계값: pushup >= 38, run_time <= 9.6, sleep_quality >= 5.2, stress_level <= 5.8
회복일 평균: fit 5.50, unfit 10.07 | 테스트 fit 예측 1/21 (0.048, train 0.125)


,soldier_id,status,recovery_days
0,S002,unfit,10.071429
1,S003,unfit,10.071429
2,S024,unfit,10.071429
3,S025,unfit,10.071429
4,S027,unfit,10.071429


In [3]:
check = pd.read_csv('submission.csv')
assert check.columns.tolist() == sample.columns.tolist()
assert check.iloc[:,0].tolist() == sample.iloc[:,0].tolist()
assert check.iloc[:,0].is_unique and not check.isna().any().any()
print('제출 형식 확인:', check.shape)


제출 형식 확인: (21, 3)
